In [23]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
# model = "claude-sonnet-4-5"
model = "claude-haiku-4-5"

In [24]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [25]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 2,
    "allowed_domains": ["nih.gov"],
}

In [26]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle? search web before you respond
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_01VZW6cJ7yP3FLDUZqkkU63C', container=None, content=[ServerToolUseBlock(id='srvtoolu_01UMZ2XGXCraSe47bcxsmtS1', caller=None, input={'query': 'best exercise building leg muscle'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='Er8lCioIDhgCIiQzZDE4YjBjOS1kOTg0LTRjNjQtOTQ1NS05YjE2YjlmNzQyOTESDDIPtxib2m+1/QgaphoMCb82ZBuydI1cqgV4IjAtiW+Cpb5VOv3lzjFhQ0PC4f1N3qULjkgNukqYlY2DepcE7HVSXIfbzpNUFIgNLOoqwiRV5bS0VFWHn4iG/vUrP1/RPJyRa1jdANMLCjImDKZC2ihuAH0k47YoRUIzExk/4vtT5YRx/pS8MeiKMq4Bz8grpK3JdbKc1eFOSyNfI1skACPEg2HVvcYZCeTYZ5lp8PSp/FWL7uxIY6HqkAgvvegRYCthn9ReAotOumcEtsAPwigdwXd+k5n5fIH6zXmi79eY7WKuX9d6hHt+6tK8aeQmm+GgQoFHN5nQzHREovD2imk8REmhia9t+NRfilalTgUxp2ZUbRryiamhDo6S3FsoEo1CszdS26XhweIcaW5gXS97RemnL2MZFdXpaLTMTV658K8iZcAurXS/YI0i6vmFUu8hBYx9FOh/tlGRGwm8cgeWOM1c5ddk1ulrOwPLRtoqW0sBApT+hi2sr0aAHUkkFNqzyuAvxrNRMdMJ6Q8qGq88vC7d4okeQ3WyCeyrMdAHSFgK8AcibfU1gMTqLq3rEjSEWcIERDlC

In [28]:
# deep dive into each content block...

response.content[0]

ServerToolUseBlock(id='srvtoolu_01UMZ2XGXCraSe47bcxsmtS1', caller=None, input={'query': 'best exercise building leg muscle'}, name='web_search', type='server_tool_use')

In [ ]:
# WebSearchToolResultBlock - web_search_result (with title, URL, etc.)

response.content[1].content[0]

WebSearchResultBlock(encrypted_content='Er8lCioIDhgCIiQzZDE4YjBjOS1kOTg0LTRjNjQtOTQ1NS05YjE2YjlmNzQyOTESDDIPtxib2m+1/QgaphoMCb82ZBuydI1cqgV4IjAtiW+Cpb5VOv3lzjFhQ0PC4f1N3qULjkgNukqYlY2DepcE7HVSXIfbzpNUFIgNLOoqwiRV5bS0VFWHn4iG/vUrP1/RPJyRa1jdANMLCjImDKZC2ihuAH0k47YoRUIzExk/4vtT5YRx/pS8MeiKMq4Bz8grpK3JdbKc1eFOSyNfI1skACPEg2HVvcYZCeTYZ5lp8PSp/FWL7uxIY6HqkAgvvegRYCthn9ReAotOumcEtsAPwigdwXd+k5n5fIH6zXmi79eY7WKuX9d6hHt+6tK8aeQmm+GgQoFHN5nQzHREovD2imk8REmhia9t+NRfilalTgUxp2ZUbRryiamhDo6S3FsoEo1CszdS26XhweIcaW5gXS97RemnL2MZFdXpaLTMTV658K8iZcAurXS/YI0i6vmFUu8hBYx9FOh/tlGRGwm8cgeWOM1c5ddk1ulrOwPLRtoqW0sBApT+hi2sr0aAHUkkFNqzyuAvxrNRMdMJ6Q8qGq88vC7d4okeQ3WyCeyrMdAHSFgK8AcibfU1gMTqLq3rEjSEWcIERDlCJDX+exGtGOQRgFRqY12Z+d2aPAdBL4RiCjUBq/9GHkT+LTON2CaXXWQJ7wvyZTzsD+VQS5QReybFaISgHbLBjCS9rKrYGIm3583U+gI5is77Fy0ofmLIuO3TGftEg1LIevFF9YMf8fFErYrKV+zibDZD8+fjtoGJ4FgHOXMeBXDKR+uuX5uCALmQ3tCkmWs5XVb20OcEVLe0APqzJflgmECN2XXfgZsseREwRP2Hy7lGa7GIXrSFtKqJ9ZtAbv5xmbEsYQCfWF5tlCzSgiTAwjWuBXToDvoV53+GHke+3Zv/iGX9AKED

In [53]:
response.content[1].content[9]

WebSearchResultBlock(encrypted_content='EtsbCioIDhgCIiQzZDE4YjBjOS1kOTg0LTRjNjQtOTQ1NS05YjE2YjlmNzQyOTESDDXlV0/pEW5/Z4xFKhoMCjsMN5TveKDiElX6IjCVOKZZOkshqmPMcAIEkiD2rAj5zIWD1f76b71yiEd1IpSFf3EciJh4Dq7j136+eWoq3hqb7P20ccThHzGR+BeLqRK6s1S71Va0I8vHCvwrGB/e3+3JScrOXFiZJrP6/a2CdDdXfPRXNO7haFzTFQ5/cjeLfBxp+isplMb1ZcvRS3NqBQWwgZ8qpGjNWB/y4/OMbqjOiCXDqiDVlm0P4Ss4z36mAVSoa72nzwe8VtEzk+Z9m7jTt6+EaMp475nRAX/L4MjpD/wzH4dV0rMSHWTRyYIHk/XqlLQptmOIpoyHChIoA37Uq1eMRKyOfLAkjjt+WlzzYDOWFaMsTOw2aZGYb7eBF5aouWC3LGJg+Am07svdwM74N7cU8duHKDRgpBH6rAtcy53obbrlOhLu93e/WnMO4TX+0s0vm5MWOh8x8WgvteGa01byi9mXScSFO3tihExoN7CKOuhjYXO/sHs3J7pAAmLiCmMLC54me6ddMdUouW8iM3I0vKxflJCGkZEArJD4PzIbkl48WIvx0VqDd9CSmTTbbL9UVPxawGZQrnG5LNF0j3wyWVpAE8D1wkvHQiV0zFEwpy2g5KBRSFa1ZX0gSEwnQz9NPlox9VVbhEcaGzye83DvuFf4979LxQZ+9JuuJjqI2q0HljBRNW4NPBoBCl2sDk/RYIvZQv897FBvVc88OUhCXvZRZE8dV9Wy4kRregScTlfEqGPoJkM5P4NOrAaGF86gUQUG9A2/i4lp321C6kXG7mgs28q+1XQXyHt/dhehRcMQqD6cXef3CpZa4GqZka5Eek4PeH/078UldPjUUYqOXQ8aaZlUmh6sEeDcuPJsgYPLEm7ITS9DWlLl

In [42]:
response.content[2]

TextBlock(citations=None, text='Based on research, there isn\'t a single "best" exercise for leg muscle gain—effectiveness depends on your training approach. However, several exercises consistently show strong results:\n\n**Most Effective Exercises:**\n\n', type='text')

In [ ]:
# CitationsWebSearchResultLocation - text

response.content[3].text

'The leg press is one of the most typical exercises for strengthening the lower limbs.'

In [ ]:
# CitationsWebSearchResultLocation - web_search_result_location (with title, url, cited_text, etc.)

response.content[3].citations

[CitationsWebSearchResultLocation(cited_text='The leg press is one of the most typical exercises for strengthening the lower limbs. ', encrypted_index='Eo8BCioIDhgCIiQzZDE4YjBjOS1kOTg0LTRjNjQtOTQ1NS05YjE2YjlmNzQyOTESDCDj5ZWjaJIsiGCvHhoMSt2kHN1oC5B0Nbl7IjB0oIFMGWN0jofvO2ovQzVnBjulBEIWTGTz0/0pzKAl6JfvtkzFS/56uHCumTocRngqE+ZoOuzxRNImA6FPTdaFPIvmQhcYBA==', title='Influence of Feet Position and Execution Velocity on Muscle Activation and Kinematic Parameters During the Inclined Leg Press Exercise - PMC', type='web_search_result_location', url='https://pmc.ncbi.nlm.nih.gov/articles/PMC9112713/')]

In [44]:
response.content[4]

TextBlock(citations=None, text=' ', type='text')

In [65]:
response.content[12].text

"\n\n**Bottom Line:**\n\nAny of these exercises—leg press, squats, or deadlifts—combined with progressive resistance training (gradually increasing weight or volume), proper form, and consistency will effectively build leg muscle. Choose the one you can perform safely with good form and that you'll stick with long-term."

In [67]:
text_from_message(response)

'Based on research, there isn\'t a single "best" exercise for leg muscle gain—effectiveness depends on your training approach. However, several exercises consistently show strong results:\n\n**Most Effective Exercises:**\n\n\nThe leg press is one of the most typical exercises for strengthening the lower limbs.\n \nLeg strengthening exercises targeting the hip extensors, hip abductors, knee extensors, and ankle plantarflexors are effective for building muscle.\n\n\n\nCompound weightlifting exercises, such as the conventional squat and deadlift, place large amounts of load and tension at the hip and lumbar spine to increase both muscle mass and strength.\n\n\n**Key Factors for Success:**\n\nThe exercise itself is only part of the equation. \nVolume, expressed as the number of sets performed, is an important driver of muscle hypertrophy, with an established linear dose-response relationship.\n \nResistance training with moderate to heavy loads and different movement speeds leads to increa

In [69]:
from IPython.display import Markdown, display

display(Markdown(text_from_message(response)))

Based on research, there isn't a single "best" exercise for leg muscle gain—effectiveness depends on your training approach. However, several exercises consistently show strong results:

**Most Effective Exercises:**


The leg press is one of the most typical exercises for strengthening the lower limbs.
 
Leg strengthening exercises targeting the hip extensors, hip abductors, knee extensors, and ankle plantarflexors are effective for building muscle.



Compound weightlifting exercises, such as the conventional squat and deadlift, place large amounts of load and tension at the hip and lumbar spine to increase both muscle mass and strength.


**Key Factors for Success:**

The exercise itself is only part of the equation. 
Volume, expressed as the number of sets performed, is an important driver of muscle hypertrophy, with an established linear dose-response relationship.
 
Resistance training with moderate to heavy loads and different movement speeds leads to increases in muscle strength and functional performance.


**Bottom Line:**

Any of these exercises—leg press, squats, or deadlifts—combined with progressive resistance training (gradually increasing weight or volume), proper form, and consistency will effectively build leg muscle. Choose the one you can perform safely with good form and that you'll stick with long-term.